<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ocp_load_pretrained_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Use OCP pretrained models for JARVIS-Leaderboard

In [ ]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-cluster -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

In [ ]:
import os
if not os.path.exists('ocp'):
     !git clone https://github.com/Open-Catalyst-Project/ocp.git
os.chdir('ocp')
!pip install -e .

In [ ]:
pip -q install  lmdb pymatgen==2023.5.10 ase==3.22.1 orjson e3nn==0.2.6 wandb

In [ ]:
import os, torch
from ase.io import read
from ocpmodels.preprocessing import AtomsToGraphs
from ocpmodels.models import SchNet

In [ ]:
!wget https://dl.fbaipublicfiles.com/opencatalystproject/models/2021_02/is2re/schnet_all.pt

In [ ]:
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
attrs = torch.load("schnet_all.pt", map_location=device)["config"][
    "model_attributes"
]

In [ ]:
model = SchNet(
    None,
    None,
    1,
    cutoff=6.0,
    hidden_channels=attrs["hidden_channels"],
    num_interactions=attrs["num_interactions"],
    num_gaussians=attrs["num_gaussians"],
    regress_forces=False,
    use_pbc=True,
)

state_dict = torch.load("schnet_all.pt", map_location=device)["state_dict"]
state_dict = {k[2 * len("module.") :]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=True)
model.to(device)

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -e .

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.core.specie import atomic_numbers_to_symbols
import os, torch
from ase.io import read
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from jarvis.core.atoms import Atoms
from tqdm import tqdm
from jarvis.db.figshare import  get_request_data
import json,zipfile
import numpy as np
import pandas as pd
from jarvis.db.jsonutils import loadjson,dumpjson
import os
from jarvis.core.atoms import Atoms
from jarvis.core.atoms import Atoms
from jarvis.core.specie import atomic_numbers_to_symbols
from ocpmodels.datasets import SinglePointLmdbDataset
import os, torch
from ase.io import read
from ocpmodels.preprocessing import AtomsToGraphs
from ocpmodels.models import CGCNN,SchNet
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from jarvis.core.atoms import Atoms
from ocpmodels.datasets import data_list_collater
from jarvis.db.figshare import data
import pandas as pd
dat = data('ocp10k')
df = pd.DataFrame(dat)

path = "/content/jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/SinglePropertyPrediction/ocp10k_relaxed_energy.json.zip"
js_tag = "ocp10k_relaxed_energy.json"
id_data = json.loads(zipfile.ZipFile(path).read(js_tag))
train_ids = np.array(list(id_data['train'].keys()))
val_ids = np.array(list(id_data['val'].keys()))
test_ids = np.array(list(id_data['test'].keys()))
train_df = df[df['id'].isin(train_ids)]
val_df = (df[df['id'].isin(val_ids)])#[:take_val]
test_df = (df[df['id'].isin(test_ids)])

In [ ]:
model.eval()

In [ ]:
%%time
a2g = AtomsToGraphs(
    max_neigh=200,
    radius=6,
    r_energy=False,
    r_forces=False,
    r_distances=True,
    r_edges=True,
)
target_mean= -1.525913953781128
target_std= 2.279365062713623
f=open('AI-SinglePropertyPrediction-relaxed_energy-ocp10k-test-mae.csv','w')
f.write('id,target,prediction\n')

for ii,i in tqdm(test_df.iterrows()):
    fname=i['id']
    atoms=(Atoms.from_dict(i['atoms'])).ase_converter()
    actual=i['relaxed_energy']
    relaxed_energy = (actual-target_mean)/target_std
    scaled=relaxed_energy
    data = a2g.convert(atoms).to(device)
    batch = data_list_collater([data], otf_graph=False)
    out = model(batch)
    pred=(out[0].detach().cpu().numpy().flatten().tolist()[0])*target_std+target_mean
    line=str(fname)+','+str(actual)+','+str(pred) #+'\n'
    #line=str(i.sid)+','+str(actual)+','+str(scaled)+','+str(pred) #+'\n'
    f.write(line+'\n')
f.close()

In [ ]:
from ocpmodels.common.relaxation.ase_utils import OCPCalculator
import os
os.chdir('/content/ocp')
!wget -q https://dl.fbaipublicfiles.com/opencatalystproject/models/2021_08/s2ef/gemnet_t_direct_h512_all.pt
checkpoint_path = "/content/ocp/gemnet_t_direct_h512_all.pt"
config_yml_path = "configs/s2ef/all/gemnet/gemnet-dT.yml"

# Define the calculator
calc = OCPCalculator(config_yml=config_yml_path, checkpoint=checkpoint_path)

In [ ]:
for ii,i in tqdm(test_df.iterrows()):
    fname=i['id']
    atoms=(Atoms.from_dict(i['atoms'])).ase_converter()
    actual=i['relaxed_energy']
    relaxed_energy = (actual-target_mean)/target_std
    scaled=relaxed_energy
    data = a2g.convert(atoms).to(device)
    batch = data_list_collater([data], otf_graph=False)
    out = model(batch)
    pred=(out[0].detach().cpu().numpy().flatten().tolist()[0])*target_std+target_mean
    atoms.calc=calc
    pred2=atoms.get_potential_energy()
    #pred2=pred2*target_std+target_mean
    print(fname,actual,pred,pred2)
    break